In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

test = pd.read_csv('/content/ChurnZero_test_v1.csv')
train = pd.read_csv(
    '/content/ChurnZero_dataset_v1.csv',
    on_bad_lines='skip'
)

## 1. Exploratory Data Analysis & Feature Engineering

In [2]:

# ==========================================
# 1. DEFINE COLUMN GROUPS
# ==========================================
TARGET = 'churn'
DROP_COLS = ['customer_id']
# ==========================================
# 1. DEFINE COLUMN GROUPS
# ==========================================
TARGET = 'churn'
DROP_COLS = ['customer_id']

# Categorical columns to encode
categorical_features = [
    'gender', 'marital_status', 'education_level', 'occupation_type',
    'income_band', 'income_category', 'city_tier', 'region',
    'customer_segment', 'onboarding_channel', 'relationship_type',
    'primary_account_type', 'card_category', 'customer_feedback_sentiment'
]

# Ensure DROP_COLS and TARGET are strictly excluded from numerical features
numeric_features = [
    col for col in train.columns
    if col not in categorical_features + [TARGET] + DROP_COLS
]


In [3]:

# ==========================================
# 2. FEATURE ENGINEERING FUNCTION
# ==========================================
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates high-signal domain features (ratios and interactions).
    """
    df = df.copy()

    # Financial Stress & Liquidity Indicators
    df['balance_to_income_ratio'] = df['current_balance'] / (df['annual_income'] + 1e-5)
    df['credit_utilization_spread'] = df['credit_utilization_ratio'] - df['credit_utilization_6m_avg']

    # Digital vs In-Branch Engagement Ratios
    df['digital_vs_branch_ratio'] = df['total_digital_logins'] / (df['branch_visit_count'] + 1)

    # Retention & Friction Signal
    df['unresolved_complaint_ratio'] = df['unresolved_complaint_count'] / (df['total_complaints'] + 1e-5)
    df['campaign_conversion_rate'] = df['campaign_response_count'] / (df['campaign_received_count'] + 1e-5)

    # Skewed feature log transformations
    skewed_features = ['annual_income', 'total_trans_amt', 'customer_lifetime_value', 'current_balance']
    for col in skewed_features:
        if col in df.columns:
            df[f'{col}_log'] = np.log1p(np.maximum(0, df[col]))

    return df

# Apply Feature Engineering
train_fe = engineer_features(train)
test_fe = engineer_features(test)

# Update numerical columns list with newly engineered features
NEW_NUMERICAL_COLS = [
    col for col in train_fe.columns
    if col not in categorical_features + [TARGET] + DROP_COLS
]


In [4]:

# ==========================================
# 3. BUILD PREPROCESSING PIPELINE
# ==========================================
# Numerical pipeline: Impute missing values with median, then scale
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: Impute missing values with mode, then One-Hot Encode
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Full Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ]
)


In [5]:
# ==========================================
# 4. PREPARE TRAIN AND TEST FEATURES
# ==========================================

X_train = train_fe.drop(columns=[TARGET] + DROP_COLS)
y_train = train_fe[TARGET]

X_test = test_fe.drop(columns=DROP_COLS)




numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()


print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)



from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)




X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)




encoded_cat_cols = (
    preprocessor
    .named_transformers_["cat"]
    .named_steps["onehot"]
    .get_feature_names_out(categorical_features)
    .tolist()
)

all_feature_names = numeric_features + encoded_cat_cols




X_train_final = pd.DataFrame(
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed,
    columns=all_feature_names,
    index=X_train.index
)

X_test_final = pd.DataFrame(
    X_test_processed.toarray()
    if hasattr(X_test_processed, "toarray")
    else X_test_processed,
    columns=all_feature_names,
    index=X_test.index
)


print("X_train_final shape:", X_train_final.shape)
print("X_test_final shape:", X_test_final.shape)

Numeric features:
['age', 'dependent_count', 'annual_income', 'tenure_months', 'number_of_products', 'customer_lifetime_value', 'loyalty_program_member', 'referral_count', 'last_contacted_days', 'relationship_manager_assigned', 'avg_monthly_balance', 'current_balance', 'balance_decline_percentage', 'monthly_transaction_count', 'monthly_transaction_value', 'cash_withdrawal_count', 'upi_transaction_count', 'debit_card_transaction_count', 'net_banking_transaction_count', 'account_inactive_days', 'total_trans_amt', 'total_trans_count', 'total_amt_chng_q4_q1', 'total_ct_chng_q4_q1', 'avg_open_to_buy', 'total_revolving_bal', 'savings_account_flag', 'current_account_flag', 'credit_card_flag', 'personal_loan_flag', 'home_loan_flag', 'auto_loan_flag', 'fixed_deposit_flag', 'investment_product_flag', 'insurance_product_flag', 'demat_account_flag', 'credit_card_limit', 'credit_card_spend', 'credit_utilization_ratio', 'minimum_due_paid_flag', 'late_credit_card_payment_count', 'loan_outstanding_amo

In [6]:

# ==========================================
# 5. CALCULATE CLASS IMBALANCE FOR XGBOOST
# ==========================================
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight_value = neg_count / pos_count

print("Phase 2 Execution Complete!")
print(f"Processed Train Shape: {X_train_final.shape}")
print(f"Processed Test Shape : {X_test_final.shape}")
print(f"Calculated scale_pos_weight for XGBoost: {scale_pos_weight_value:.2f}")

Phase 2 Execution Complete!
Processed Train Shape: (8101, 156)
Processed Test Shape : (2026, 156)
Calculated scale_pos_weight for XGBoost: 5.22


## 2. Model Training, Tuning & Evaluation

In [7]:
import os
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    precision_recall_curve,
    auc,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score
)
from xgboost import XGBClassifier

# Create output folder for artifacts
os.makedirs('models', exist_ok=True)


In [8]:

# ==========================================
# 1. SPLIT LOCAL TRAIN & VALIDATION SETS
# ==========================================
# Split X_train_final and y_train into local train (80%) and validation (20%)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=42
)

print(f"Local Train Shape : {X_tr.shape}, Target Positive Rate: {y_tr.mean():.2%}")
print(f"Validation Shape  : {X_val.shape}, Target Positive Rate: {y_val.mean():.2%}")


Local Train Shape : (6480, 156), Target Positive Rate: 16.06%
Validation Shape  : (1621, 156), Target Positive Rate: 16.10%


In [9]:

# ==========================================
# 2. EVALUATION FUNCTION
# ==========================================
def evaluate_predictions(y_true, y_probs, threshold=0.50, model_name="Model"):
    """Evaluates probabilities against true labels."""
    y_preds = (y_probs >= threshold).astype(int)

    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)

    print(f"\n=== {model_name} Evaluation (Threshold = {threshold:.2f}) ===")
    print(f"PR-AUC (Primary Metric) : {pr_auc:.4f}")
    print(f"F1-Score                : {f1_score(y_true, y_preds):.4f}")
    print(f"ROC-AUC                 : {roc_auc_score(y_true, y_probs):.4f}")
    print(f"Precision               : {precision_score(y_true, y_preds, zero_division=0):.4f}")
    print(f"Recall                  : {recall_score(y_true, y_preds):.4f}")
    print("\nClassification Report:\n", classification_report(y_true, y_preds, zero_division=0))

    return pr_auc, y_probs


In [10]:

# ==========================================
# 3. BASELINE: LOGISTIC REGRESSION
# ==========================================
print("\n--- Step 1: Baseline Logistic Regression ---")

baseline_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
baseline_model.fit(X_tr, y_tr)

val_probs_baseline = baseline_model.predict_proba(X_val)[:, 1]
evaluate_predictions(y_val, val_probs_baseline, model_name="Logistic Regression (Validation)")



--- Step 1: Baseline Logistic Regression ---

=== Logistic Regression (Validation) Evaluation (Threshold = 0.50) ===
PR-AUC (Primary Metric) : 0.9674
F1-Score                : 0.8783
ROC-AUC                 : 0.9929
Precision               : 0.8137
Recall                  : 0.9540

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.96      0.97      1360
           1       0.81      0.95      0.88       261

    accuracy                           0.96      1621
   macro avg       0.90      0.96      0.93      1621
weighted avg       0.96      0.96      0.96      1621



/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


(np.float64(0.967388719135929),
 array([7.17153382e-04, 3.28926556e-05, 3.69411161e-02, ...,
        1.15652686e-01, 9.89040237e-01, 1.66375297e-02]))

In [11]:

# ==========================================
# 4. XGBOOST WITH CROSS-VALIDATION
# ==========================================
print("\n--- Step 2: Stratified 5-Fold Cross-Validation on Local Train ---")

scale_pos_weight_value = (y_tr == 0).sum() / (y_tr == 1).sum()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_pr_aucs = []

for fold, (train_idx, fold_val_idx) in enumerate(cv.split(X_tr, y_tr)):
    X_fold_tr, y_fold_tr = X_tr.iloc[train_idx], y_tr.iloc[train_idx]
    X_fold_val, y_fold_val = X_tr.iloc[fold_val_idx], y_tr.iloc[fold_val_idx]

    fold_model = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight_value,
        eval_metric='aucpr',
        random_state=42,
        n_jobs=-1
    )
    fold_model.fit(X_fold_tr, y_fold_tr)

    fold_probs = fold_model.predict_proba(X_fold_val)[:, 1]
    p, r, _ = precision_recall_curve(y_fold_val, fold_probs)
    fold_pr_auc = auc(r, p)
    cv_pr_aucs.append(fold_pr_auc)
    print(f"Fold {fold + 1} PR-AUC: {fold_pr_auc:.4f}")

print(f"Mean CV PR-AUC: {np.mean(cv_pr_aucs):.4f} (+/- {np.std(cv_pr_aucs):.4f})")

# Fit final XGBoost on entire local training split
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_value,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_tr, y_tr)

# Evaluate on Validation set
val_probs_xgb = xgb_model.predict_proba(X_val)[:, 1]
pr_auc_xgb, _ = evaluate_predictions(y_val, val_probs_xgb, threshold=0.50, model_name="XGBoost (Validation - Default Threshold)")



--- Step 2: Stratified 5-Fold Cross-Validation on Local Train ---
Fold 1 PR-AUC: 0.9999
Fold 2 PR-AUC: 1.0000
Fold 3 PR-AUC: 1.0000
Fold 4 PR-AUC: 1.0000
Fold 5 PR-AUC: 1.0000
Mean CV PR-AUC: 1.0000 (+/- 0.0000)

=== XGBoost (Validation - Default Threshold) Evaluation (Threshold = 0.50) ===
PR-AUC (Primary Metric) : 0.9999
F1-Score                : 0.9942
ROC-AUC                 : 1.0000
Precision               : 1.0000
Recall                  : 0.9885

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      1360
           1       1.00      0.99      0.99       261

    accuracy                           1.00      1621
   macro avg       1.00      0.99      1.00      1621
weighted avg       1.00      1.00      1.00      1621



In [12]:

# ==========================================
# 5. OPTIMIZE THRESHOLD ON VALIDATION SET
# ==========================================
print("\n--- Step 3: Threshold Tuning for Maximum F1-Score ---")

precision, recall, thresholds = precision_recall_curve(y_val, val_probs_xgb)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = float(thresholds[best_idx])

print(f"Optimal Decision Threshold found on Validation Set: {best_threshold:.4f}")
evaluate_predictions(y_val, val_probs_xgb, threshold=best_threshold, model_name="XGBoost (Validation - Tuned Threshold)")



--- Step 3: Threshold Tuning for Maximum F1-Score ---
Optimal Decision Threshold found on Validation Set: 0.3309

=== XGBoost (Validation - Tuned Threshold) Evaluation (Threshold = 0.33) ===
PR-AUC (Primary Metric) : 0.9999
F1-Score                : 0.9962
ROC-AUC                 : 1.0000
Precision               : 1.0000
Recall                  : 0.9923

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      1360
           1       1.00      0.99      1.00       261

    accuracy                           1.00      1621
   macro avg       1.00      1.00      1.00      1621
weighted avg       1.00      1.00      1.00      1621



(np.float64(0.999898541494167),
 array([8.6782675e-04, 5.5911416e-05, 2.7603444e-04, ..., 1.9931783e-04,
        9.9878365e-01, 2.1129787e-04], dtype=float32))

In [13]:

# ==========================================
# 6. RETRAIN ON FULL TRAIN DATA & PREDICT ON TEST
# ==========================================
print("\n--- Step 4: Retraining on 100% of Train Data & Predicting on Test ---")

# Train final production model using ALL of X_train_final
full_scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

final_production_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=full_scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)
final_production_model.fit(X_train_final, y_train)

# Predict churn probabilities on the unseen test set (X_test_final)
test_churn_probabilities = final_production_model.predict_proba(X_test_final)[:, 1]
test_churn_predictions = (test_churn_probabilities >= best_threshold).astype(int)

# Create submission / predictions dataframe
test_results_df = pd.DataFrame({
    'churn_probability': test_churn_probabilities,
    'predicted_churn': test_churn_predictions
})

print("\nSample Test Set Predictions:")
print(test_results_df.head())



--- Step 4: Retraining on 100% of Train Data & Predicting on Test ---

Sample Test Set Predictions:
   churn_probability  predicted_churn
0           0.000034                0
1           0.000354                0
2           0.000520                0
3           0.000042                0
4           0.000242                0


In [14]:

# ==========================================
# 7. SAVE ARTIFACTS
# ==========================================
joblib.dump(preprocessor, 'models/preprocessor.pkl')
joblib.dump(final_production_model, 'models/xgboost_model.pkl')

metadata = {
    'optimal_threshold': best_threshold,
    'feature_names': all_feature_names,
    'scale_pos_weight': full_scale_pos_weight
}
joblib.dump(metadata, 'models/model_metadata.pkl')

print("\nSaved artifacts successfully to models/ directory.")


Saved artifacts successfully to models/ directory.


In [15]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
import os

project_path = '/content/drive/MyDrive/ChurnZero-Customer-Churn-Prediction'

os.makedirs(project_path, exist_ok=True)

%cd /content/drive/MyDrive/ChurnZero-Customer-Churn-Prediction

print("Project folder:")
print(os.getcwd())

/content/drive/MyDrive/ChurnZero-Customer-Churn-Prediction
Project folder:
/content/drive/MyDrive/ChurnZero-Customer-Churn-Prediction


In [35]:
!mkdir -p ~/.ssh

!ssh-keygen -t ed25519 \
    -C "hopeagain502@gmail.com" \
    -f ~/.ssh/github_colab \
    -N ""

Generating public/private ed25519 key pair.
Your identification has been saved in /root/.ssh/github_colab
Your public key has been saved in /root/.ssh/github_colab.pub
The key fingerprint is:
SHA256:MVYwcS3QeIgjRh2FECaVNb7QRnXgZ8BhtLbwXgK9VE4 hopeagain502@gmail.com
The key's randomart image is:
+--[ED25519 256]--+
|  .o**=O&XE.     |
|   o++==+@+ .    |
|   ...*.X.+.     |
|     o O B       |
|      . S .      |
|       . o       |
|        .        |
|                 |
|                 |
+----[SHA256]-----+


In [36]:
!ssh-keyscan github.com >> ~/.ssh/known_hosts

!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/github_colab
!chmod 644 ~/.ssh/github_colab.pub

# github.com:22 SSH-2.0-af8ca74
# github.com:22 SSH-2.0-af8ca74
# github.com:22 SSH-2.0-af8ca74
# github.com:22 SSH-2.0-af8ca74
# github.com:22 SSH-2.0-af8ca74


In [37]:
import subprocess
import os

agent_output = subprocess.check_output(
    ['ssh-agent', '-s'],
    text=True
)

for line in agent_output.splitlines():
    if line.startswith('SSH_AUTH_SOCK'):
        key, value = line.split(';')[0].split('=')
        os.environ[key] = value

    elif line.startswith('SSH_AGENT_PID'):
        key, value = line.split(';')[0].split('=')
        os.environ[key] = value

subprocess.run(
    ['ssh-add', os.path.expanduser('~/.ssh/github_colab')],
    check=True
)

print("SSH key added successfully.")

SSH key added successfully.


In [38]:
!cat ~/.ssh/github_colab.pub

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIJCUzvwKDIs/pF00BtQxz+t44U8nePjuQxKuJpYmXNo9 hopeagain502@gmail.com


In [40]:
%%writefile ~/.ssh/config
Host github.com
    HostName github.com
    User git
    IdentityFile ~/.ssh/github_colab
    IdentitiesOnly yes

Writing /root/.ssh/config


In [41]:
!chmod 600 ~/.ssh/config
!chmod 600 ~/.ssh/github_colab

In [42]:
!ssh -T git@github.com

Hi hopeagain502-code! You've successfully authenticated, but GitHub does not provide shell access.


In [45]:
!git remote set-url origin git@github.com:hopeagain502-code/ChurnZero-Customer-Churn-Prediction.git

In [46]:
!git remote -v

origin	git@github.com:hopeagain502-code/ChurnZero-Customer-Churn-Prediction.git (fetch)
origin	git@github.com:hopeagain502-code/ChurnZero-Customer-Churn-Prediction.git (push)


In [48]:
!git branch -M main
!git push -u origin main

Enumerating objects: 3, done.
Counting objects: 100% (3/3), done.
Delta compression using up to 2 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 311 bytes | 103.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To github.com:hopeagain502-code/ChurnZero-Customer-Churn-Prediction.git
 * [new branch]      main -> main
branch 'main' set up to track 'origin/main'.


In [49]:
%cd /content/drive/MyDrive/ChurnZero-Customer-Churn-Prediction
!pwd
!find . -maxdepth 3 -type f | sort

/content/drive/MyDrive/ChurnZero-Customer-Churn-Prediction
/content/drive/MyDrive/ChurnZero-Customer-Churn-Prediction
./.git/COMMIT_EDITMSG
./.git/config
./.git/description
./.git/HEAD
./.git/hooks/applypatch-msg.sample
./.git/hooks/commit-msg.sample
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/post-update.sample
./.git/hooks/pre-applypatch.sample
./.git/hooks/pre-commit.sample
./.git/hooks/pre-merge-commit.sample
./.git/hooks/prepare-commit-msg.sample
./.git/hooks/pre-push.sample
./.git/hooks/pre-rebase.sample
./.git/hooks/pre-receive.sample
./.git/hooks/push-to-checkout.sample
./.git/hooks/sendemail-validate.sample
./.git/hooks/update.sample
./.gitignore
./.git/index
./.git/info/exclude
./.git/logs/HEAD


In [50]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [51]:
!find /content -type f \( -name "*.ipynb" -o -name "*.pkl" -o -name "*.pt" -o -name "*.pth" -o -name "*.h5" -o -name "*.joblib" \) | sort

/content/drive/MyDrive/Colab Notebooks/00_pytorch_fundamental.ipynb
/content/drive/MyDrive/Colab Notebooks/01_pytorch_workflow (1).ipynb
/content/drive/MyDrive/Colab Notebooks/02_pytorch_classification_ (1).ipynb
/content/drive/MyDrive/Colab Notebooks/02_pytorch_classification_.ipynb
/content/drive/MyDrive/Colab Notebooks/03_pytorch_computer_vision (1) (1).ipynb
/content/drive/MyDrive/Colab Notebooks/03_pytorch_computer_vision (1).ipynb
/content/drive/MyDrive/Colab Notebooks/03_pytorch_computer_vision (2).ipynb
/content/drive/MyDrive/Colab Notebooks/03_pytorch_computer_vision.ipynb
/content/drive/MyDrive/Colab Notebooks/04_PyTorch_Custom_Datasets (1).ipynb
/content/drive/MyDrive/Colab Notebooks/04_PyTorch_Custom_Datasets.ipynb
/content/drive/MyDrive/Colab Notebooks/asl_chatbot (1).ipynb
/content/drive/MyDrive/Colab Notebooks/asl-chatbot.ipynb
/content/drive/MyDrive/Colab Notebooks/asl_chatbot.ipynb
/content/drive/MyDrive/Colab Notebooks/asl_webcam.ipynb
/content/drive/MyDrive/Colab Not

In [52]:
%cd /content/drive/MyDrive/ChurnZero-Customer-Churn-Prediction

!mkdir -p notebooks models

/content/drive/MyDrive/ChurnZero-Customer-Churn-Prediction


In [53]:
!ls -lh "/content/drive/MyDrive/Colab Notebooks/End-to-End-E-Commerce-Customer-Churn-Retention-System.ipynb"

-rw------- 1 root root 38K Sep 23 08:09 '/content/drive/MyDrive/Colab Notebooks/End-to-End-E-Commerce-Customer-Churn-Retention-System.ipynb'
